# Post-SPP RL Admission Filter: Colab Training Stage

Correct pipeline:

```text
cluster:
  generate candidate table input

Colab:
  train policy on candidate table
  save policy table / held-out replay summary
  git push trained result

cluster:
  pull Colab-trained policy
  patch ChampSim spp_dev to query the policy before prefetch_line
  replay real IPC / hit rate / miss rate
```

This notebook is only the Colab training stage. It does not run ChampSim.


## 0. Input and output

Input active table:

```text
projects/post_prefetch_filter/data/generated/spp_candidate_log.csv.xz
```

Saved output:

```text
projects/post_prefetch_filter/results/colab_feature_sweeps/<trace>_<scope>/
  candidate_sanity.csv
  policy_replay_summary.csv
  feature_sweep_compare_vs_F0.csv
  policy_tables.json
  best_policy.json
  best_policy_admit_states.csv
  run_manifest.json
  interpretation.md
```


## 1. Clone or update repo


In [ ]:
from pathlib import Path
import os, subprocess

GITHUB_USERNAME = 'Angelawoo572'
REPO_NAME = 'cache_arch'
REPO_ROOT = Path('/content') / REPO_NAME
REPO_URL = f'https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git'

if not REPO_ROOT.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'reset', '--hard', 'origin/main'], check=True)

os.chdir(REPO_ROOT)
subprocess.run(['git', 'log', '--oneline', '-5'], check=True)
subprocess.run(['git', 'status', '--short'], check=True)


## 2. Train policy in Colab and save artifacts


In [ ]:
from pathlib import Path
import os, subprocess, pandas as pd

REPO_ROOT = Path('/content/cache_arch')
os.chdir(REPO_ROOT)
PROJECT = REPO_ROOT / 'projects' / 'post_prefetch_filter'
INPUT = PROJECT / 'data' / 'generated' / 'spp_candidate_log.csv.xz'
OUT_ROOT = PROJECT / 'results' / 'colab_feature_sweeps'

CANDIDATE_SCOPE = 'spp_l2_issue'
MAX_ROWS = 300_000
TRAIN_FRAC = 0.80
MIN_CONFIDENCE = 90
STATE_LIMIT = 50_000

assert INPUT.exists(), f'Missing active input: {INPUT}'

cmd = [
    'python3', 'projects/post_prefetch_filter/scripts/06_colab_train_policy.py',
    '--input', str(INPUT),
    '--out-root', str(OUT_ROOT),
    '--scope', CANDIDATE_SCOPE,
    '--max-rows', str(MAX_ROWS),
    '--train-frac', str(TRAIN_FRAC),
    '--min-confidence', str(MIN_CONFIDENCE),
    '--state-limit', str(STATE_LIMIT),
]
print('running:', ' '.join(cmd))
subprocess.run(cmd, check=True)
run_dirs = sorted([p for p in OUT_ROOT.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime, reverse=True)
RUN_DIR = run_dirs[0]
print('RUN_DIR =', RUN_DIR)
summary = pd.read_csv(RUN_DIR / 'policy_replay_summary.csv')
display(summary)
compare = pd.read_csv(RUN_DIR / 'feature_sweep_compare_vs_F0.csv')
display(compare[['feature_set','issued_ratio','accuracy','useful_kept_ratio','bad_suppressed_ratio','estimated_reward','delta_estimated_reward_vs_F0']])
print((RUN_DIR / 'interpretation.md').read_text())


## 3. Push Colab-trained artifacts

If this cell asks for authentication, use your normal GitHub/Colab workflow. It commits only the generated run directory.


In [ ]:
import os, subprocess
from pathlib import Path

REPO_ROOT = Path('/content/cache_arch')
os.chdir(REPO_ROOT)
rel_run_dir = str(RUN_DIR.relative_to(REPO_ROOT))
subprocess.run(['git', 'pull', '--rebase', 'origin', 'main'], check=True)
subprocess.run(['git', 'add', rel_run_dir], check=True)
subprocess.run(['git', 'status', '--short'], check=True)
commit_msg = f'Save Colab-trained policy {RUN_DIR.name}'
diff = subprocess.run(['git', 'diff', '--cached', '--quiet'])
if diff.returncode == 0:
    print('No new Colab result changes to commit.')
else:
    subprocess.run(['git', 'commit', '-m', commit_msg], check=True)
    subprocess.run(['git', 'push', 'origin', 'main'], check=True)
    print('pushed:', commit_msg)


## 4. Cluster next step

After this notebook pushes results:

```bash
cd /scratch/qianruw/cache
git pull
```

Use these artifacts for ChampSim replay:

```text
projects/post_prefetch_filter/results/colab_feature_sweeps/<trace>_spp_l2_issue/best_policy.json
projects/post_prefetch_filter/results/colab_feature_sweeps/<trace>_spp_l2_issue/best_policy_admit_states.csv
```

Final metrics to compare against original SPP: IPC, L1D/L2C/LLC hit rate, miss rate, MPKI, and prefetch issued/useful/useless/accuracy.
